## Section 1: Generic/collective entity prevalence

In [1]:
import pandas as pd
import ast

df = pd.read_parquet("../data/arf_chunks_parsed.parquet")

# Pick a book you already know well for easy eyeballing.
BOOK_ID = "106"
book_df = df[df["book_id"] == BOOK_ID].sort_values("chunk_id").reset_index(drop=True)

print(f"{len(book_df)} chunks for book {BOOK_ID}")
print(f"chunk_id range: {book_df['chunk_id'].min()} - {book_df['chunk_id'].max()}")

# Re-confirm contiguity for *this* book before relying on it here —
# README notes 19/20 sampled books were contiguous, book 106 wasn't
# one of the checked ones as far as this session's notes show.
ids = book_df["chunk_id"].astype(int).tolist()
gaps = [(a, b) for a, b in zip(ids, ids[1:]) if b - a > 1]
print(f"Gaps in chunk_id sequence: {len(gaps)} gaps found"
      f"{f' (showing first 5): {gaps[:5]}' if gaps else ''}")

# Reconstruct a short contiguous passage from adjacent chunks.
# NOTE: chunks are 5-sentence rolling windows with 1-sentence overlap,
# so naive concatenation duplicates ~1 sentence per boundary. Fine for
# a feasibility smoke test; not fine for anything built on this later.
N_CHUNKS = 15
START_IDX = 0
passage_chunks = book_df.iloc[START_IDX:START_IDX + N_CHUNKS]
passage_text = " ".join(passage_chunks["chunk"].tolist())
print(f"\nPassage length: {len(passage_text.split())} words")
preview_words = passage_text.split()[:60]
print(" ".join(preview_words), "...")
# Ground truth check: how many raw entity strings in THIS passage's
# relations already look pronoun-like or generically-descriptive,
# rather than a proper name? This is the real-data version of
# README's "his brother" caveat.
PRONOUN_LIKE = {
    "he", "she", "it", "they", "him", "her", "them", "his", "hers",
    "their", "himself", "herself", "itself", "themselves",
}

def looks_generic(name: str) -> bool:
    tokens = name.lower().split()
    if not tokens:
        return False
    if tokens[0] in PRONOUN_LIKE:
        return True
    # crude heuristic: starts with a determiner/possessive and no token
    # in the string is capitalized (i.e. no proper noun present)
    if tokens[0] in {"the", "his", "her", "their", "a", "an"} and not any(
        t[0].isupper() for t in name.split()
    ):
        return True
    return False

generic_mentions = set()
for rels in passage_chunks["relations"]:
    parsed = ast.literal_eval(rels) if isinstance(rels, str) else rels
    if not parsed:
        continue
    for r in parsed:
        for key in ("entity1", "entity2"):
            name = r.get(key, "")
            if looks_generic(name):
                generic_mentions.add(name)

print(f"\n{len(generic_mentions)} generic/pronoun-like raw entity strings found:")
for m in sorted(generic_mentions):
    print(f"  - {m!r}")

883 chunks for book 106
chunk_id range: 0 - 99
Gaps in chunk_id sequence: 88 gaps found (showing first 5): [(1, 10), (10, 100), (11, 110), (12, 120), (13, 130)]

Passage length: 1340 words
*** JUNGLE TALES OF TARZAN *** [Illustration] Jungle Tales of Tarzan by Edgar Rice Burroughs Contents CHAPTER I. Tarzan's First Love CHAPTER II. The Capture of Tarzan CHAPTER III. The Fight for the Balu CHAPTER IV. The God of Tarzan CHAPTER V. Tarzan and the Black Boy CHAPTER VI. The Witch-Doctor Seeks Vengeance CHAPTER VII. The Witch-Doctor Seeks Vengeance CHAPTER ...

2 generic/pronoun-like raw entity strings found:
  - 'he'
  - 'the elephant'


In [2]:
# How many rows actually share each chunk_id, for this book?
dup_counts = book_df["chunk_id"].value_counts().sort_index()
print(dup_counts.head(15))
print(f"\nmax rows sharing one chunk_id: {dup_counts.max()}")
print(f"chunk_id dtype: {book_df['chunk_id'].dtype}")

# And: where does actual Chapter I narrative text start?
first_chapter_idx = book_df[book_df["chunk"].str.contains(
    "CHAPTER I\\b", regex=True, na=False
)].index
print(f"\nRows possibly containing a 'CHAPTER I' marker: {list(first_chapter_idx)[:5]}")

chunk_id
0      1
1      1
10     1
100    1
101    1
102    1
103    1
104    1
105    1
106    1
107    1
108    1
109    1
11     1
110    1
Name: count, dtype: int64

max rows sharing one chunk_id: 1
chunk_id dtype: str

Rows possibly containing a 'CHAPTER I' marker: [0, 112]


In [3]:
book_df["chunk_id"] = book_df["chunk_id"].astype(int)
book_df = book_df.sort_values("chunk_id").reset_index(drop=True)

ids = book_df["chunk_id"].tolist()
gaps = [(a, b) for a, b in zip(ids, ids[1:]) if b - a > 1]
print(f"True chunk_id range: {ids[0]} - {ids[-1]}")
print(f"Gaps in true numeric order: {gaps if gaps else 'none'}")

first_chapter_idx = book_df[book_df["chunk"].str.contains(
    r"CHAPTER I\b", regex=True, na=False
)].index
print(f"Rows containing 'CHAPTER I' (true order): {list(first_chapter_idx)[:5]}")

START_IDX = first_chapter_idx[0] + 3   # skip past the heading row(s)

N_CHUNKS = 60   # ~25 rolling 5-sentence windows — big enough to cross 
                # paragraph/scene boundaries, the actual thing worth testing

passage_chunks = book_df.iloc[START_IDX:START_IDX + N_CHUNKS]
passage_text = " ".join(passage_chunks["chunk"].tolist())

print(f"\nPassage length: {len(passage_text.split())} words")
print(f"Chunk id range covered: {passage_chunks['chunk_id'].min()} - {passage_chunks['chunk_id'].max()}")
print(passage_text[:500], "...")

True chunk_id range: 0 - 882
Gaps in true numeric order: none
Rows containing 'CHAPTER I' (true order): [0, 2]

Passage length: 6613 words
Chunk id range covered: 3 - 62
Just to have seen him there, lolling upon the swaying bough of the jungle-forest giant, his brown skin mottled by the brilliant equatorial sunlight which percolated through the leafy canopy of green above him, his clean-limbed body relaxed in graceful ease, his shapely head partly turned in contemplative absorption and his intelligent, gray eyes dreamily devouring the object of their devotion, you would have thought him the reincarnation of some demigod of old.

 You would not have guessed tha ...


## Section 2: Coreference model selection


In [4]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import logging
logging.getLogger("httpx").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("fastcoref").setLevel(logging.ERROR)

from fastcoref import FCoref
model = FCoref(device="cpu")
# fastcoref requires transformers<5.0 — v5's PreTrainedModel refactor
# (post_init() / all_tied_weights_keys) isn't compatible with fastcoref's
# created separate venv for its own transformers to avoid dependency collisions
preds = model.predict(texts=[passage_text])
clusters = preds[0].get_clusters()

print(f"{len(clusters)} coreference clusters found:\n")
for cluster in clusters:
    print(cluster)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

c:\Users\baaqa\Downloads\Narrative Intelligence\narrative-intelligence\.venv-coref\Lib\site-packages\pyarrow\compute.py:230: FutureWarning: Specifying null_placement in SortOptions is deprecated as of 25.0.0. Specify null_placement per sort_key instead.
  return options_class(*args, **kwargs)


Inference:   0%|          | 0/1 [00:00<?, ?it/s]

137 coreference clusters found:

['him', 'his', 'him', 'his', 'his', 'his', 'him', 'he', 'his', 'his', 'he', 'Teeka', 'his', 'he', 'his', 'Teeka', 'Teeka', 'her', 'Teeka', 'Teeka', 'her', 'Teeka', 'Teeka', 'her', 'the young female', 'she', 'her', "Teeka's", "Teeka's", "Teeka's", "Teeka's", 'her', 'her', "Teeka's", "Teeka's", 'her', 'her', 'Teeka', 'her', "Teeka's", 'Teeka', 'Taug', 'the young she', 'her', 'Taug', 'Taug', 'Teeka', 'Taug', 'Taug', 'his', 'his', 'Taug', 'he', 'he', 'Taug', 'Teeka', 'Taug', 'Teeka', 'the she', 'he', 'his', 'his', 'his', 'Taug', 'his', 'Teeka', 'herself', 'Teeka', 'herself', 'she', 'his', 'she', 'she', 'Taug', 'his', 'Teeka', 'she', 'Taug', 'Teeka', 'she', 'Taug', 'the young bull', 'Taug', 'he', 'Taug', 'his', 'His', 'His', 'He', 'Teeka', 'Teeka', "Taug's", 'his', 'his', 'he', "Taug's", 'the ape-boy', "Taug's", 'the ape-boy', 'he', 'his', 'Taug', 'he', 'Teeka', 'her', 'her', "Teeka's", 'She', 'she', 'her', 'she', 'her', 'she', 'her', 'her', 'she', 'her', 'h

fastcoref at N_CHUNKS=60 (~6,600 words): failed once with RuntimeError
("bad allocation"), succeeded on a later cold-kernel rerun with
identical code and input. Confirmed NOT a PyTorch allocator warm-up
effect (reproduced on a fresh kernel). Most likely transient system
memory pressure unrelated to this code. Treat 60 chunks as within
this machine's memory ceiling but close enough to it that failures
are possible depending on concurrent system load — not a stable,
guaranteed-safe size. Relevant to Week 4 Stage B: production runs
over full books need either explicit windowing or a comfortable
memory margin, not just "it worked on my machine once." 

In [5]:
from fastcoref import LingMessCoref

model = LingMessCoref(device="cpu")
# failed
# preds = model.predict(texts=[passage_text])   # same 60-chunk passage that succeeded with FCoref
# clusters = preds[0].get_clusters()
short_passage = " ".join(book_df.iloc[3:20]["chunk"].tolist())  # ~17 chunks, much shorter
preds = model.predict(texts=[short_passage])
print(len(preds))
if preds:
    print(preds[0].get_clusters())

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Inference:   0%|          | 0/1 [00:00<?, ?it/s]

1
[['him', 'his', 'him', 'his', 'his', 'his', 'him', 'he', 'his', 'his', 'he', 'the ape-man', 'his', 'he', 'his', 'Tarzan of the Apes', 'his', 'Tarzan of the Apes', 'his', 'he', 'he', 'Tarzan', 'Tarzan', 'Tarzan', 'Tarzan', 'he', 'his', 'he', 'he', 'himself', 'he', "Tarzan's", 'Tarzan', 'his', 'his', 'his', 'he', 'his', 'his', 'his', 'He', 'His', 'he', 'he', 'he', 'his', 'he', "Tarzan's", "Tarzan's", 'Tarzan', 'his', 'his', 'he', 'his', 'he', 'he', 'he', 'Tarzan', 'his', 'his', 'he', 'Tarzan', 'Tarzan', 'Tarzan', 'his', 'Tarzan', 'Tarzan', 'the young ape-man', 'the young ape-man', 'He', 'Tarzan of the Apes', 'he', 'Tarzan', 'Tarzan', 'his', 'Tarzan', 'he', 'Tarzan', 'his', 'Tarzan', 'he', 'Tarzan', 'his', 'his', 'Tarzan of the Apes', 'His', 'his', 'he', 'his', "Tarzan's", 'the ape-man', 'his', "Tarzan's", 'his', 'he', 'Tarzan', 'The latter', 'his', 'his', 'Tarzan', 'the ape-boy', 'His', 'his', 'he', 'he'], ['his intelligent, gray eyes', 'their'], ['the Apes', 'the Apes'], ['the truth o

LingMessCoref (Longformer-large backbone) returned an EMPTY preds list
(IndexError on preds[0]) on the same ~6,613-word / 60-chunk passage that
FCoref succeeded on. Hypothesis: passage exceeds Longformer's ~4,096-token
ceiling and is silently dropped rather than erroring clearly — confirmed/
refuted by rerunning on a much shorter passage (see cell above). If
confirmed: LingMess trades F-coref's segment-and-stitch flexibility for a
harder length ceiling, despite its attention mechanism being cheaper
per-token. Relevant to Week 4 Stage B — neither model handles a full-book
pass without explicit windowing.